# Day 2: Streaming Ingestion (Kafka / MSK + Spark Streaming)

This notebook simulates streaming ingestion by generating synthetic FHIR data and appending it to the Bronze table.

In [ ]:
import os
import time
import json
import uuid
import threading
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit

spark = SparkSession.builder \
    .appName("Day02_Streaming_Ingestion") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.4_2.12:1.4.1") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "/tmp/iceberg_warehouse") \
    .getOrCreate()


In [ ]:
spark.sql("""
CREATE DATABASE IF NOT EXISTS local.ehdip_bronze_db
""")
spark.sql("""
CREATE TABLE IF NOT EXISTS local.ehdip_bronze_db.raw_payloads (
    ingestion_timestamp TIMESTAMP,
    source_system_id STRING,
    payload_id STRING,
    raw_payload_json STRING
)
USING iceberg
PARTITIONED BY (days(ingestion_timestamp))
""")

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
from pyspark.sql.functions import col

# Create memory stream
spark.conf.set("spark.sql.streaming.schemaInference", True)

schema = StructType([
    StructField("payload_id", StringType(), True),
    StructField("raw_payload_json", StringType(), True)
])

# A rate stream for simulation
df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 1) \
    .load()

# Transform
streaming_df = df.selectExpr(
    "CAST(value AS STRING) AS payload_id",
    "'{\"resourceType\": \"Patient\", \"id\": \"' || value || '\"}' AS raw_payload_json"
) \
.withColumn("ingestion_timestamp", current_timestamp()) \
.withColumn("source_system_id", lit("memory_stream"))

# Write stream
query = streaming_df.writeStream \
    .format("iceberg") \
    .outputMode("append") \
    .trigger(processingTime="5 seconds") \
    .option("checkpointLocation", "/tmp/checkpoints/streaming_fhir") \
    .toTable("local.ehdip_bronze_db.raw_payloads")

import time
time.sleep(15)
query.stop()
print("Streaming finished.")

spark.sql("SELECT * FROM local.ehdip_bronze_db.raw_payloads LIMIT 5").show(truncate=False)
